# Signal Architecture Visualization

This notebook explores the signal architecture for device health classification:
- **Healthy signals**: Gaussian peaks (centered, narrow, high amplitude, low noise)
- **Unhealthy signals**: Lorentzian peaks (off-center, wide, low amplitude, high noise)

We'll visualize:
1. Individual signal types with different parameters
2. Peak features (height, SNR, FWHM)
3. Feature separation between healthy/unhealthy
4. Signal decomposition (base + noise)

In [ ]:
# Import required libraries
import sys

sys.path.append("..")

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.signal_processing.feature_extractor import extract_features
from src.signal_processing.signal_generator import (
    add_gaussian_noise,
    create_time_array,
    generate_gaussian_peak,
    generate_lorentzian_peak,
)
from src.signal_processing.signal_models import SignalData

plotly_template = "plotly_dark"

## 1. Healthy Gaussian Signals

Healthy signals have:
- **Center**: μ ∈ [48, 52] (well-centered)
- **Width**: σ ∈ [2, 3] (narrow peaks)
- **Height**: 2.5 - 3.0 (high amplitude)
- **Noise**: 1-2% (low noise)

In [ ]:
# Generate healthy Gaussian signals with different parameters
time = create_time_array()

# Three variations of healthy signals
healthy_signals = []
params = [
    {"mu": 48.0, "sigma": 2.5, "height": 2.8, "noise": 0.015, "label": "Boundary (μ=48)"},
    {"mu": 50.0, "sigma": 2.3, "height": 2.9, "noise": 0.012, "label": "Centered (μ=50)"},
    {"mu": 52.0, "sigma": 2.7, "height": 2.6, "noise": 0.018, "label": "Boundary (μ=52)"},
]

for p in params:
    base_signal = generate_gaussian_peak(time, p["mu"], p["sigma"], p["height"])
    signal_with_noise = add_gaussian_noise(base_signal, p["noise"])
    healthy_signals.append(
        {"time": time, "amplitude": signal_with_noise, "base": base_signal, "params": p}
    )

# Plot healthy signals
fig = make_subplots(
    rows=1, cols=3, subplot_titles=[p["label"] for p in params], horizontal_spacing=0.1
)

for i, signal in enumerate(healthy_signals):
    fig.add_trace(
        go.Scatter(
            x=signal["time"],
            y=signal["amplitude"],
            mode="lines",
            name=signal["params"]["label"],
            line=dict(color="green", width=2),
            hovertemplate=(
                f"Time: %{{x}}<br>"
                f"Amplitude: %{{y:.3f}}<br>"
                f"μ: {signal['params']['mu']}<br>"
                f"σ: {signal['params']['sigma']}<br>"
                f"Height: {signal['params']['height']}<br>"
                f"Noise: {signal['params']['noise'] * 100:.1f}%"
            ),
            showlegend=False,
        ),
        row=1,
        col=i + 1,
    )

fig.update_xaxes(title_text="Time", row=1, col=2)
fig.update_yaxes(title_text="Amplitude", row=1, col=1)
fig.update_layout(
    title="Healthy Gaussian Signals (μ ∈ [48, 52], σ ∈ [2, 3])",
    height=400,
    showlegend=False,
    template=plotly_template,
)
fig.show()

## 2. Unhealthy Lorentzian Signals

Unhealthy signals have:
- **Center**: μ ∈ [42, 47] ∪ [53, 58] (off-center)
- **Width**: γ ∈ [4.5, 6] (wide peaks)
- **Height**: 1.0 - 1.5 (low amplitude)
- **Noise**: 6-10% (high noise)

In [ ]:
# Generate unhealthy Lorentzian signals
unhealthy_signals = []
params_unhealthy = [
    {"mu": 43.0, "gamma": 5.2, "height": 1.2, "noise": 0.08, "label": "Off-center left (μ=43)"},
    {"mu": 45.5, "gamma": 4.8, "height": 1.4, "noise": 0.07, "label": "Shifted left (μ=45.5)"},
    {"mu": 55.0, "gamma": 5.5, "height": 1.1, "noise": 0.09, "label": "Off-center right (μ=55)"},
]

for p in params_unhealthy:
    base_signal = generate_lorentzian_peak(time, p["mu"], p["gamma"], p["height"])
    signal_with_noise = add_gaussian_noise(base_signal, p["noise"])
    unhealthy_signals.append(
        {"time": time, "amplitude": signal_with_noise, "base": base_signal, "params": p}
    )

# Plot unhealthy signals
fig = make_subplots(
    rows=1, cols=3, subplot_titles=[p["label"] for p in params_unhealthy], horizontal_spacing=0.1
)

for i, signal in enumerate(unhealthy_signals):
    fig.add_trace(
        go.Scatter(
            x=signal["time"],
            y=signal["amplitude"],
            mode="lines",
            name=signal["params"]["label"],
            line=dict(color="red", width=2),
            hovertemplate=(
                f"Time: %{{x}}<br>"
                f"Amplitude: %{{y:.3f}}<br>"
                f"μ: {signal['params']['mu']}<br>"
                f"γ: {signal['params']['gamma']}<br>"
                f"Height: {signal['params']['height']}<br>"
                f"Noise: {signal['params']['noise'] * 100:.1f}%"
            ),
            showlegend=False,
        ),
        row=1,
        col=i + 1,
    )

fig.update_xaxes(title_text="Time", row=1, col=2)
fig.update_yaxes(title_text="Amplitude", row=1, col=1)
fig.update_layout(
    title="Unhealthy Lorentzian Signals (μ ∈ [42, 47] ∪ [53, 58], γ ∈ [4.5, 6])",
    height=400,
    showlegend=False,
    template=plotly_template,
)
fig.show()

## 3. Overlay: Healthy vs Unhealthy

Let's overlay typical healthy and unhealthy signals to see the visual differences.

In [ ]:
# Create overlay plot
fig = go.Figure()

# Healthy signal (centered, narrow, high)
healthy = generate_gaussian_peak(time, mu=50.0, sigma=2.5, height=2.8)
healthy_noisy = add_gaussian_noise(healthy, noise_level=0.015)

# Unhealthy signal (off-center, wide, low)
unhealthy = generate_lorentzian_peak(time, mu=45.0, gamma=5.0, height=1.2)
unhealthy_noisy = add_gaussian_noise(unhealthy, noise_level=0.08)

fig.add_trace(
    go.Scatter(
        x=time,
        y=healthy_noisy,
        mode="lines",
        name="Healthy (Gaussian μ=50, σ=2.5)",
        line=dict(color="green", width=2),
    )
)

fig.add_trace(
    go.Scatter(
        x=time,
        y=unhealthy_noisy,
        mode="lines",
        name="Unhealthy (Lorentzian μ=45, γ=5)",
        line=dict(color="red", width=2),
    )
)

fig.update_layout(
    title="Healthy vs Unhealthy Signal Comparison",
    xaxis_title="Time",
    yaxis_title="Amplitude",
    height=500,
    hovermode="x unified",
    template=plotly_template,
)

fig.show()

## 4. Peak Features Extraction

Extract quantitative features from signals:
- **Peak Height**: Maximum amplitude
- **Signal-to-Noise Ratio (SNR)**: Signal strength relative to noise
- **Full Width at Half Maximum (FWHM)**: Peak width
- **Noise Level**: Standard deviation of noise

In [ ]:
# Extract features from all signals
import pandas as pd

feature_data = []

# Process healthy signals
for signal in healthy_signals:
    signal_data = SignalData(
        time=signal["time"], amplitude=signal["amplitude"], shape_type="gaussian"
    )
    features = extract_features(signal_data)
    feature_data.append(
        {
            "label": "Healthy",
            "mu": signal["params"]["mu"],
            "peak_height": features["peak_height"],
            "snr": features["snr"],
            "fwhm": features["fwhm"],
            "noise_level": features["noise_level"],
        }
    )

# Process unhealthy signals
for signal in unhealthy_signals:
    signal_data = SignalData(
        time=signal["time"], amplitude=signal["amplitude"], shape_type="lorentzian"
    )
    features = extract_features(signal_data)
    feature_data.append(
        {
            "label": "Unhealthy",
            "mu": signal["params"]["mu"],
            "peak_height": features["peak_height"],
            "snr": features["snr"],
            "fwhm": features["fwhm"],
            "noise_level": features["noise_level"],
        }
    )

df_features = pd.DataFrame(feature_data)
print("Feature Statistics by Label:")
print(df_features.groupby("label")[["peak_height", "snr", "fwhm", "noise_level"]].mean())

## 5. Feature Separation Visualization

Visualize how features separate healthy from unhealthy signals.

In [ ]:
# Create 2x2 feature scatter plots
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Peak Height vs SNR",
        "FWHM vs Noise Level",
        "Peak Height vs FWHM",
        "SNR vs Noise Level",
    ),
    horizontal_spacing=0.15,
    vertical_spacing=0.15,
)

# Plot 1: Peak Height vs SNR
for label, color in [("Healthy", "green"), ("Unhealthy", "red")]:
    data = df_features[df_features["label"] == label]
    fig.add_trace(
        go.Scatter(
            x=data["peak_height"],
            y=data["snr"],
            mode="markers",
            name=label,
            marker=dict(size=12, color=color),
            showlegend=True,
        ),
        row=1,
        col=1,
    )

# Plot 2: FWHM vs Noise Level
for label, color in [("Healthy", "green"), ("Unhealthy", "red")]:
    data = df_features[df_features["label"] == label]
    fig.add_trace(
        go.Scatter(
            x=data["fwhm"],
            y=data["noise_level"],
            mode="markers",
            name=label,
            marker=dict(size=12, color=color),
            showlegend=False,
        ),
        row=1,
        col=2,
    )

# Plot 3: Peak Height vs FWHM
for label, color in [("Healthy", "green"), ("Unhealthy", "red")]:
    data = df_features[df_features["label"] == label]
    fig.add_trace(
        go.Scatter(
            x=data["peak_height"],
            y=data["fwhm"],
            mode="markers",
            name=label,
            marker=dict(size=12, color=color),
            showlegend=False,
        ),
        row=2,
        col=1,
    )

# Plot 4: SNR vs Noise Level
for label, color in [("Healthy", "green"), ("Unhealthy", "red")]:
    data = df_features[df_features["label"] == label]
    fig.add_trace(
        go.Scatter(
            x=data["snr"],
            y=data["noise_level"],
            mode="markers",
            name=label,
            marker=dict(size=12, color=color),
            showlegend=False,
        ),
        row=2,
        col=2,
    )

# Update axes labels
fig.update_xaxes(title_text="Peak Height", row=1, col=1)
fig.update_yaxes(title_text="SNR", row=1, col=1)

fig.update_xaxes(title_text="FWHM", row=1, col=2)
fig.update_yaxes(title_text="Noise Level", row=1, col=2)

fig.update_xaxes(title_text="Peak Height", row=2, col=1)
fig.update_yaxes(title_text="FWHM", row=2, col=1)

fig.update_xaxes(title_text="SNR", row=2, col=2)
fig.update_yaxes(title_text="Noise Level", row=2, col=2)

fig.update_layout(
    title="Feature Separation: Healthy vs Unhealthy Signals",
    height=700,
    template=plotly_template,
)

fig.show()

## 6. Signal Decomposition

Visualize signal = base + noise components.

In [ ]:
# Decompose a healthy signal
time = create_time_array()
base_healthy = generate_gaussian_peak(time, mu=50.0, sigma=2.5, height=2.8)
noise_healthy = np.random.normal(0, 0.015, len(time))
signal_healthy = base_healthy + noise_healthy

# Decompose an unhealthy signal
base_unhealthy = generate_lorentzian_peak(time, mu=45.0, gamma=5.0, height=1.2)
noise_unhealthy = np.random.normal(0, 0.08, len(time))
signal_unhealthy = base_unhealthy + noise_unhealthy

# Create decomposition plots
fig = make_subplots(
    rows=2,
    cols=1,
    subplot_titles=("Healthy Signal Decomposition", "Unhealthy Signal Decomposition"),
    vertical_spacing=0.15,
)

# Healthy decomposition
fig.add_trace(
    go.Scatter(
        x=time,
        y=base_healthy,
        mode="lines",
        name="Base (Gaussian)",
        legendgroup="Healthy",
        showlegend=True,
        line=dict(color="darkgreen", width=2),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=time,
        y=noise_healthy,
        mode="lines",
        name="Noise (σ=0.015)",
        legendgroup="Healthy",
        showlegend=True,
        line=dict(color="lightgreen", width=1),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=time,
        y=signal_healthy,
        mode="lines",
        name="Signal (Base + Noise)",
        legendgroup="Healthy",
        showlegend=True,
        line=dict(color="green", width=2, dash="dot"),
    ),
    row=1,
    col=1,
)

# Unhealthy decomposition (legendgroup and unique names)
fig.add_trace(
    go.Scatter(
        x=time,
        y=base_unhealthy,
        mode="lines",
        name="Base (Lorentzian)",
        legendgroup="Unhealthy",
        showlegend=True,
        line=dict(color="darkred", width=2),
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=time,
        y=noise_unhealthy,
        mode="lines",
        name="Noise (σ=0.08)",
        legendgroup="Unhealthy",
        showlegend=True,
        line=dict(color="lightcoral", width=1),
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=time,
        y=signal_unhealthy,
        mode="lines",
        name="Signal (Base + Noise)",
        legendgroup="Unhealthy",
        showlegend=True,
        line=dict(color="red", width=2, dash="dot"),
    ),
    row=2,
    col=1,
)

fig.update_layout(height=700, hovermode="x unified", template=plotly_template, showlegend=True)
fig.show()

## Summary

**Key Observations:**
1. **Healthy signals** have higher peak heights (2.6-2.9) and SNR (>15)
2. **Unhealthy signals** have lower peak heights (1.1-1.4) and SNR (<10)
3. **FWHM** is narrower for healthy (~5) vs unhealthy (~10)
4. **Noise levels** are 4-5x higher in unhealthy signals

These features provide clear separation for classification!